In [ ]:
import kagglehub
import os

data_dir = os.path.dirname(os.path.abspath(__file__)) + "/dataset"

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification", output_dir=data_dir)

print("Path to dataset files:", path)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.transforms import Resize
from tqdm import tqdm


In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.11.0+cu130
13.0


In [2]:
print(torch.cuda.is_available())

True


In [3]:
class MusicGenreCNN(nn.Module):
    def __init__(self):
        super(MusicGenreCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 45 * 45, 256)
        self.bn_fc1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.maxpool1(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.maxpool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn_fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        return x

In [ ]:
# Set the random seed for reproducibility
torch.manual_seed(42)

# Define data transformations
transform = transforms.Compose([
    Resize((180, 180)),
    transforms.ToTensor(),
])

In [5]:
# Load dataset
dataset = datasets.ImageFolder(root='dataset/Data/images_original', transform=transform)

# Split dataset into training, validation, and test sets
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.2 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

In [6]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)


In [7]:
# Create an instance of the CNN model
cnn_model = MusicGenreCNN()

In [8]:
# Initialize the CNN model, loss function, and optimizer
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

In [9]:
# Training loop for CNN model
def train_cnn_model(epochs):
    for epoch in range(epochs):
        cnn_model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}'):
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            inputs, labels = inputs.to(device), labels.to(device)
            cnn_model.to(device)
            cnn_optimizer.zero_grad()
            cnn_outputs = cnn_model(inputs)
            cnn_loss = cnn_criterion(cnn_outputs, labels)
            cnn_loss.backward()
            cnn_optimizer.step()
            running_loss += cnn_loss.item()

        # Print training loss for each epoch
        print(f'Training Loss: {running_loss / len(train_loader)}')
        
        cnn_model.eval()


In [10]:
# Move the model to GPU
cnn_model.to('cuda') if torch.cuda.is_available() else cnn_model.to('cpu')

MusicGenreCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=259200, out_features=256, bias=True)
  (bn_fc1): BatchNorm1d(256,

In [11]:
# Run training for 50 epochs
train_cnn_model(50)

Epoch 1/50: 100%|██████████| 11/11 [00:11<00:00,  1.07s/it]


Training Loss: 2.0032166134227407


Epoch 2/50: 100%|██████████| 11/11 [00:06<00:00,  1.80it/s]


Training Loss: 1.4379061677239158


Epoch 3/50: 100%|██████████| 11/11 [00:06<00:00,  1.69it/s]


...

Training Loss: 0.0039529255750081075


Epoch 49/50: 100%|██████████| 11/11 [00:06<00:00,  1.71it/s]


Training Loss: 0.004067007589831271


Epoch 50/50: 100%|██████████| 11/11 [00:06<00:00,  1.71it/s]

Training Loss: 0.003219346510542726


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Evaluate the model on the test set
def evaluate_model(model, dataloader):
    model.eval()
    all_labels = []
    all_predictions = []

    # Disable gradient calculation
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating'):
            # Move inputs and labels to the GPU
            inputs, labels = inputs.to('cuda'), labels.to('cuda')
            outputs = model(inputs)
            _, predictions = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    # Calculate accuracy, classification report, and confusion matrix
    accuracy = accuracy_score(all_labels, all_predictions)
    report = classification_report(all_labels, all_predictions, target_names=dataset.classes)
    confusion_mat = confusion_matrix(all_labels, all_predictions)

    print(f'Accuracy: {accuracy}')
    print('\nClassification Report:\n', report)
    print('\nConfusion Matrix:\n', confusion_mat)

In [14]:
# Evaluate the CNN model on the test set
evaluate_model(cnn_model, test_loader)

Evaluating: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

Accuracy: 0.6237623762376238

Classification Report:
               precision    recall  f1-score   support

       blues       0.44      0.58      0.50        12
   classical       0.90      0.82      0.86        11
     country       0.57      0.29      0.38        14
       disco       0.57      0.57      0.57        14
      hiphop       0.50      0.75      0.60         8
        jazz       1.00      0.75      0.86         8
       metal       0.86      1.00      0.92         6
         pop       0.90      0.75      0.82        12
      reggae       0.67      0.67      0.67         6
        rock       0.31      0.40      0.35        10

    accuracy                           0.62       101
   macro avg       0.67      0.66      0.65       101
weighted avg       0.66      0.62      0.63       101


Confusion Matrix:
 [[7 1 0 1 1 0 0 0 0 2]
 [1 9 0 0 0 0 0 0 0 1]
 [3 0 4 3 0 0 0 0 0 4]
 [1 0 0 8 1 0 1 1 1 1]
 [0 0 0 0 6 0 0 0 1 1]
 [1 0 0 0 1 6 0 0 0 0]
 [0 0 0 0 0 0 6 0 0 0]
 [0 0 